# Silver Layer - Products Processing

This notebook processes CDC events from the Bronze layer and merges them into the Silver products table with schema evolution support and **schema drift detection**.

In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
%run ../helpers/NB_schema_contracts

In [ ]:
from pyspark.sql.functions import col, coalesce, expr, from_json, to_timestamp, unbase64, row_number, current_timestamp, lit
from pyspark.sql.types import IntegerType, LongType, StringType, StructField, StructType
from pyspark.sql.window import Window
import uuid

dbutils.widgets.text("CATALOG", "workspace")
dbutils.widgets.text("BRONZE_SCHEMA", "bronze")
dbutils.widgets.text("BRONZE_TABLE", "products")
dbutils.widgets.text("SILVER_SCHEMA", "silver")
dbutils.widgets.text("SILVER_TABLE", "silver_products")
dbutils.widgets.text("CHECKPOINT_PATH", "/Volumes/workspace/default/mnt/checkpoints/products_silver")
dbutils.widgets.text("MONITORING_SCHEMA", "monitoring")
dbutils.widgets.text("SCHEMA_POLICY", "additive_only")  # strict, additive_only, permissive
dbutils.widgets.text("ALERT_CHANNEL", "log")  # log, webhook, all
dbutils.widgets.text("WEBHOOK_URL", "")  # Optional Slack/Teams webhook

CONFIG = {
    "catalog": dbutils.widgets.get("CATALOG"),
    "bronze_schema": dbutils.widgets.get("BRONZE_SCHEMA"),
    "bronze_table": dbutils.widgets.get("BRONZE_TABLE"),
    "silver_schema": dbutils.widgets.get("SILVER_SCHEMA"),
    "silver_table": dbutils.widgets.get("SILVER_TABLE"),
    "checkpoint_path": dbutils.widgets.get("CHECKPOINT_PATH"),
    "monitoring_schema": dbutils.widgets.get("MONITORING_SCHEMA"),
    "schema_policy": dbutils.widgets.get("SCHEMA_POLICY"),
    "alert_channel": dbutils.widgets.get("ALERT_CHANNEL"),
    "webhook_url": dbutils.widgets.get("WEBHOOK_URL"),
}

# Generate pipeline run ID for tracking
PIPELINE_RUN_ID = str(uuid.uuid4())

bronze_table_fqn = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{CONFIG['bronze_table']}"
silver_schema_fqn = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}"
silver_table_fqn = f"{silver_schema_fqn}.{CONFIG['silver_table']}"
drift_table_fqn = f"{CONFIG['catalog']}.{CONFIG['monitoring_schema']}.schema_drift_log"

In [ ]:
# Initialize monitoring table for schema drift tracking
create_schema_drift_table(spark, CONFIG['catalog'], CONFIG['monitoring_schema'])

# Initialize schema and table using helpers
ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])

# Check if table exists and create if needed
existing_schema = get_existing_schema(spark, silver_table_fqn)
if existing_schema is None:
    create_silver_table_products(spark, silver_table_fqn)
else:
    print(f"Using existing Silver products table: {silver_table_fqn}")

In [ ]:
# ============================================================================
# SCHEMA DRIFT VALIDATION
# ============================================================================

# Get expected schema contract
expected_schema = get_schema_contract('silver.products')

# Validate the target Silver table schema against the contract
actual_schema = get_existing_schema(spark, silver_table_fqn)

# Validate schema with policy
print(f"Validating schema against contract with policy: {CONFIG['schema_policy']}")

try:
    is_valid, drift_result = validate_schema_with_policy(
        spark,
        expected_schema=expected_schema,
        actual_schema=actual_schema,
        table_name=silver_table_fqn,
        drift_table=drift_table_fqn,
        source_system="postgres_cdc",
        pipeline_run_id=PIPELINE_RUN_ID,
        policy=CONFIG['schema_policy'],
        alert_channel=CONFIG['alert_channel'],
        webhook_url=CONFIG['webhook_url'] if CONFIG['webhook_url'] else None
    )
    
    if drift_result['has_drift']:
        print(f"⚠️ Schema drift detected: {drift_result['severity']}")
        if drift_result['added_columns']:
            print(f"  Added: {drift_result['added_columns']}")
        if drift_result['removed_columns']:
            print(f"  Removed: {drift_result['removed_columns']}")
        if drift_result['type_changes']:
            print(f"  Type changes: {drift_result['type_changes']}")
    else:
        print("✅ Schema validation passed - no drift detected")
        
except SchemaDriftException as e:
    print(f"🚨 PIPELINE BLOCKED: {e}")
    dbutils.notebook.exit(f"Schema drift blocked pipeline: {e}")

In [ ]:
# Read bronze stream with schema evolution enabled
bronze_stream = (
    spark.readStream
    .option("mergeSchema", "true")
    .table(bronze_table_fqn)
)

# Schema for parsing Debezium JSON payload for products
flexible_schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("id", IntegerType()),
            StructField("product_name", StringType()),
            StructField("weight", StructType([
                StructField("scale", IntegerType()),
                StructField("value", StringType())
            ])),
            StructField("color", StringType()),
            StructField("created_at", LongType()),
            StructField("updated_at", LongType())
        ])),
        StructField("after", StructType([
            StructField("id", IntegerType()),
            StructField("product_name", StringType()),
            StructField("weight", StructType([
                StructField("scale", IntegerType()),
                StructField("value", StringType())
            ])),
            StructField("color", StringType()),
            StructField("created_at", LongType()),
            StructField("updated_at", LongType())
        ])),
        StructField("op", StringType()),
        StructField("ts_ms", LongType())
    ]))
])

parsed = bronze_stream.select(from_json(col("value"), flexible_schema).alias("data")).select("data.payload.*")

In [ ]:
# Parse and transform CDC events
events = parsed.select(
    coalesce(col("after.id"), col("before.id")).alias("id"),
    coalesce(col("after.product_name"), col("before.product_name")).alias("product_name"),
    coalesce(col("after.weight.value"), col("before.weight.value")).alias("weight_raw"),
    coalesce(col("after.weight.scale"), col("before.weight.scale")).alias("weight_scale"),
    coalesce(col("after.color"), col("before.color")).alias("color"),
    coalesce(col("after.created_at"), col("before.created_at")).alias("created_at_raw"),
    coalesce(col("after.updated_at"), col("before.updated_at")).alias("updated_at_raw"),
    col("op"),
    to_timestamp((col("ts_ms") / 1000).cast("double")).alias("event_time")
)

# Decode weight from base64 (Debezium numeric encoding)
events = events.withColumn("weight_bytes", unbase64(col("weight_raw")))
events = events.withColumn("weight_int", expr("cast(conv(hex(weight_bytes), 16, 10) as double)"))
events = events.withColumn("weight", expr("cast(weight_int / pow(10, weight_scale) as decimal(10,2))"))
events = events.withColumn("created_at", to_timestamp((col("created_at_raw") / 1000000).cast("double")))
events = events.withColumn("updated_at", to_timestamp((col("updated_at_raw") / 1000000).cast("double")))

events = events.select(
    "id",
    "product_name",
    "weight",
    "color",
    "created_at",
    "updated_at",
    "op",
    "event_time"
)

In [ ]:
def upsert_to_silver(batch_df, batch_id):
    """Upsert CDC events into Silver products table with schema evolution."""
    if not batch_df.take(1):
        return

    # Deduplicate: keep latest event per record
    window = Window.partitionBy("id").orderBy(col("event_time").desc())
    latest = (
        batch_df
        .withColumn("rn", row_number().over(window))
        .filter(col("rn") == 1)
        .drop("rn")
    )

    # Get existing columns for schema evolution
    existing_columns = get_existing_columns(spark, silver_table_fqn)
    
    # Build dynamic merge clauses
    core_fields = ["id", "product_name", "weight", "color", "created_at", "updated_at"]
    update_set, insert_values = build_merge_clauses(
        existing_columns, 
        core_fields,
        include_inserted_dt=True,
        include_updated_dt=True
    )

    # Execute merge using helper
    delta_table = DeltaTable.forName(spark, silver_table_fqn)
    execute_merge(spark, delta_table, latest, update_set, insert_values)

In [ ]:
# Start streaming write with schema evolution
query = (
    events.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", CONFIG["checkpoint_path"])
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print(f"✅ Silver products processing completed. Run ID: {PIPELINE_RUN_ID}")